# Module 8: Operating at Scale

**Workshop Track:** 300-Level  
**Prerequisites:** Modules 1 through 7 complete

---

You have made it to the final module. Over the past seven modules you have gone from a first `fit()` call to async pipelines, resilient retry logic, and full diagnostic instrumentation. What remains is the operational layer — the things that matter not when you are building the pipeline, but when you are running it for the hundredth time.

Module 8 covers two topics that every team eventually has to solve as their NEXUS usage matures: API key management across environments, and model lifecycle management — tagging models with metadata, looking them up by ID, listing your inventory, and pruning the ones you no longer need.

None of this requires writing new algorithms. It is about discipline: having the right key in the right environment, tagging models so future-you can find them, and keeping your account inventory clean.

## Learning Objectives

By the end of this notebook you will:

- Manage API keys across development, staging, and production environments
- Give each environment its own `api_url` so the SDK connects to the right endpoint
- Use `client.models.list()` to build a complete inventory of every model trained under your API key
- Use `client.models.get(model_id)` to look up a specific model and read its attributes
- Use `client.models.set_attributes()` to tag models with version, team, stage, and ownership metadata
- Use `client.models.delete(model_id)` to prune stale models from your account
- Understand why model lifecycle management is essential for production operations

---

## Setup

In [1]:
# ============================================================================
# Workshop bootstrap — run this first. Safe to re-run. Identical in every module.
#
# In Google Colab, add two secrets via the key icon in the left sidebar
# (toggle "Notebook access" on for each):
#   • FUNDAMENTAL_API_KEY            — your NEXUS API key (ak_...)
#   • CLOUDSMITH_FUNDAMENTAL_TOKEN   — token to install the Fundamental SDK
# Run the modules in order, in a single Colab runtime: they pass state to each
# other through workshop_colab/_workshop_state.json. See README.md.
# ============================================================================
import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Fundamental-Technologies/introduction-to-nexus.git"
WORKSHOP_SUBDIR = "workshop_colab"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def _looks_like_root(p):
    p = Path(p)
    return (p / "dataset").is_dir() and (p / "module_00").is_dir()


def _find_workshop_root():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if _looks_like_root(base):
            return base
        if _looks_like_root(base / WORKSHOP_SUBDIR):
            return base / WORKSHOP_SUBDIR
    return None


if IN_COLAB:
    repo_path = Path("/content") / "introduction-to-nexus"
    if not _looks_like_root(repo_path / WORKSHOP_SUBDIR):
        print("Cloning workshop repository…")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_path)], check=True)
    WORKSHOP_ROOT = repo_path / WORKSHOP_SUBDIR
else:
    WORKSHOP_ROOT = _find_workshop_root()
    if WORKSHOP_ROOT is None:
        raise RuntimeError(
            "Could not locate the workshop_colab folder. Open this notebook from "
            "inside the cloned workshop_colab tree."
        )

WORKSHOP_ROOT = Path(WORKSHOP_ROOT).resolve()
DATASET_DIR = WORKSHOP_ROOT / "dataset"
os.chdir(WORKSHOP_ROOT)


def _get_secret(name, required=True):
    val = os.getenv(name)
    if not val and IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    if required and not val:
        raise RuntimeError(
            f"Missing secret '{name}'.\n"
            "  • In Colab: open the key icon (Secrets) in the left sidebar, add a "
            f"secret named '{name}', and turn on 'Notebook access'.\n"
            f"  • Locally: export {name} in your shell before launching Jupyter.\n"
            "See README.md for details."
        )
    return val


# --- SDK install (Colab only; locally the SDK is installed during setup) ---
try:
    import fundamental  # noqa: F401
except ImportError:
    if not IN_COLAB:
        raise RuntimeError(
            "Fundamental SDK not found. Install it locally (see README.md) before running."
        )
    _token = _get_secret("CLOUDSMITH_FUNDAMENTAL_TOKEN")
    print("Installing workshop dependencies…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(WORKSHOP_ROOT / "requirements.txt")], check=True)
    print("Installing the Fundamental SDK…")
    _index = f"https://dl.cloudsmith.io/{_token}/fundamental/fundamental-client/python/simple/"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fundamental-client==0.10.0", "--extra-index-url", _index], check=True)

# --- Authentication ---
FUNDAMENTAL_API_KEY = _get_secret("FUNDAMENTAL_API_KEY")
os.environ["FUNDAMENTAL_API_KEY"] = FUNDAMENTAL_API_KEY

from fundamental import Fundamental, NEXUSClassifier, NEXUSRegressor, set_client
client = Fundamental()
set_client(client)

# --- Cross-module state (replaces IPython %store; one JSON file per workshop run) ---
STATE_FILE = WORKSHOP_ROOT / "_workshop_state.json"


def save_state(key, value):
    state = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}
    state[key] = value
    STATE_FILE.write_text(json.dumps(state, indent=2))
    print(f"Saved '{key}' to workshop state.")


def load_state(key, default=None):
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text()).get(key, default)
    return default


def require_state(key, produced_by):
    val = load_state(key)
    if val is None:
        raise RuntimeError(
            f"'{key}' is not in the workshop state file yet. Run {produced_by} in THIS "
            "Colab runtime first — modules pass state through "
            f"{STATE_FILE.name} and must be run in order. See README.md."
        )
    return val


print(f"Workshop ready. Root: {WORKSHOP_ROOT}")
print(f"API key prefix: {FUNDAMENTAL_API_KEY[:8]}…")

Workshop ready. Root: /Users/jawhny/Documents/projects/nexus_onboarding_workshop/workshop_colab
API key prefix: ak_17749…


In [2]:
import time
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

DATA_DIR = DATASET_DIR / "credit_risk"

train_raw   = pd.read_csv(f"{DATA_DIR}/borrowers_train.csv")
holdout_raw = pd.read_csv(f"{DATA_DIR}/borrowers_holdout.csv")
snapshots   = pd.read_csv(f"{DATA_DIR}/financial_snapshots.csv", parse_dates=["snapshot_date"])
assessments = pd.read_csv(f"{DATA_DIR}/credit_assessments.csv", parse_dates=["assessment_date"])
payments    = pd.read_csv(f"{DATA_DIR}/payment_events.csv", parse_dates=["payment_date"])

snapshots_latest = (
    snapshots.sort_values("snapshot_date", ascending=False)
    .drop_duplicates(subset="borrower_id", keep="first")
    [["borrower_id", "monthly_income_usd", "income_growth_pct",
      "collateral_score", "secondary_income_flag"]]
)
assessments_latest = (
    assessments.sort_values("assessment_date", ascending=False)
    .drop_duplicates(subset="borrower_id", keep="first")
    [["borrower_id", "creditworthiness_rating", "payment_behavior_score",
      "financial_stability_score", "lender_relationship_score",
      "credit_engagement_score", "debt_service_score"]]
)
payment_agg = (
    payments.groupby("borrower_id")
    .agg(
        total_payments=("payment_id", "count"),
        on_time_rate=("on_time", lambda x: (x == "Yes").mean()),
        avg_payment_usd=("amount_usd", "mean"),
    )
    .reset_index()
)

def enrich(df):
    out = df.copy()
    out = out.merge(snapshots_latest, on="borrower_id", how="left")
    out = out.merge(assessments_latest, on="borrower_id", how="left")
    out = out.merge(payment_agg, on="borrower_id", how="left")
    return out

train_enriched   = enrich(train_raw)
holdout_enriched = enrich(holdout_raw)

drop_cols    = ["borrower_id", "first_name", "last_name", "default_flag"]
all_features = [c for c in train_enriched.columns if c not in drop_cols]

# Apply the Module 3 feature prep: the account-open date becomes the numeric
# account_tenure_days feature; categoricals pass to NEXUS as-is


def add_account_tenure(df, date_col="account_open_date", ref_date="2026-01-01"):
    """Convert the account-open date into a numeric tenure feature.

    NEXUS accepts numeric, boolean, string, and categorical columns — but not
    datetime columns. So instead of parsing the date, we derive an explicit
    numeric feature: the account's age in days at a fixed reference date.
    (A fixed reference keeps the feature stable across runs.)
    """
    out = df.copy()
    out["account_tenure_days"] = (
        pd.Timestamp(ref_date) - pd.to_datetime(out[date_col])
    ).dt.days
    return out.drop(columns=[date_col])


X_train_full = add_account_tenure(train_enriched[all_features])
X_holdout_full = add_account_tenure(holdout_enriched[all_features])

# Retrieve TOP_FEATURES from the shared workshop state file, with a fallback if it is stale
TOP_FEATURES = load_state("TOP_FEATURES")
_ok = isinstance(TOP_FEATURES, (list, tuple)) and set(TOP_FEATURES).issubset(X_train_full.columns)
if not _ok:
    TOP_FEATURES = ['monthly_income_usd', 'avg_payment_usd', 'total_employment_years', 'age',
                    'secondary_income_flag', 'account_tenure_days', 'marital_status', 'collateral_score',
                    'occupation_sector', 'num_previous_lenders', 'distance_from_branch_miles',
                    'credit_engagement_score', 'financial_stability_score', 'payment_behavior_score',
                    'creditworthiness_rating']
    print("Using fallback TOP_FEATURES (Module 4 state missing or out of date).")

X_train   = X_train_full[TOP_FEATURES]
X_holdout = X_holdout_full[TOP_FEATURES]
y_train   = train_enriched["default_flag"]
y_holdout = holdout_enriched["default_flag"]

# Load the model stored in Module 6, with a fallback to fitting fresh if the id is stale
try:
    MODULE6_MODEL_ID = require_state("MODULE6_MODEL_ID", "module_06")
    CURRENT_MODEL_ID = MODULE6_MODEL_ID
    nexus = NEXUSClassifier()
    nexus.load_model(CURRENT_MODEL_ID)
    _ = nexus.predict_proba(X_holdout.head(1))
except Exception:
    print("Stored model id missing/stale — fitting a fresh model.")
    nexus = NEXUSClassifier(mode="quality")
    nexus.fit(X_train, y_train)
    CURRENT_MODEL_ID = nexus.trained_model_id_

proba = nexus.predict_proba(X_holdout)
auc   = roc_auc_score(y_holdout, proba[:, 1])
print(f"Working model loaded: {CURRENT_MODEL_ID}")
print(f"AUC: {auc:.4f}")

Working model loaded: b135ba5b-da0c-457c-8d23-a2f58505b79f
AUC: 0.9427


---

## Part 1: API Key Management

### The Multi-Environment Problem

Every team that graduates from notebook experimentation to production pipelines runs into the same problem: you need different API keys for different environments. Your development environment should not be able to affect production models. Your staging pipeline should not accidentally bill against the production account. And rotating a compromised key should not require changing code in five different places.

The NEXUS SDK solves this through the client's two constructor parameters:

- **`api_key`** -- your API key, read from `FUNDAMENTAL_API_KEY` (or an environment-specific variable) at client construction time
- **`api_url`** -- the platform endpoint, read from `FUNDAMENTAL_API_URL` if not passed explicitly, otherwise `https://api.fundamental.tech`

Give each environment its own key and its own `api_url`, and switching environments becomes a matter of changing the values you pass at client construction -- not code.

**New to environment variables?** They are named values your shell or platform sets outside the code — the standard home for secrets. On a laptop they live in your shell profile or a local `.env` file; in CI they come from the secrets store; in production they are injected by the container or orchestrator config. The one place they never live is source control.

In [3]:
# The standard pattern: one function builds the right client for the current environment

from fundamental import Fundamental, NEXUSClassifier, set_client

# Each environment maps to its own platform endpoint
ENV_API_URLS = {
    "dev":  "https://dev.api.fundamental.tech",
    "stg":  "https://stg.api.fundamental.tech",
    "prod": "https://api.fundamental.tech",
}


def build_client(env="dev"):
    """
    Build and register a Fundamental client for the given environment.

    Reads environment-specific variables:
      FUNDAMENTAL_API_KEY_DEV
      FUNDAMENTAL_API_KEY_STG
      FUNDAMENTAL_API_KEY_PROD

    Falls back to FUNDAMENTAL_API_KEY if the environment-specific
    variable is not set. Selects the matching api_url for the environment.
    """
    suffix = env.upper()
    key = (
        os.getenv(f"FUNDAMENTAL_API_KEY_{suffix}")
        or os.getenv("FUNDAMENTAL_API_KEY")
    )

    if not key:
        raise ValueError(f"No API key found for environment: {env}")

    # Each environment maps to its own endpoint
    api_url = ENV_API_URLS[env.lower()]

    client = Fundamental(api_key=key, api_url=api_url)
    set_client(client)
    return client


# The dev and stg entries above are illustrative endpoints. This workshop's API key
# targets the production platform, so we build (and register) the prod client here.
prod_client = build_client(env="prod")
prod_key = os.getenv("FUNDAMENTAL_API_KEY_PROD") or os.getenv("FUNDAMENTAL_API_KEY")
print(f"Using API key prefix: {prod_key[:8]}...")
print(f"Endpoint: {prod_client.config.api_url}")

Using API key prefix: ak_17749...
Endpoint: https://api.fundamental.tech


### Key Rotation

API key rotation is a security hygiene requirement for any production system. The NEXUS approach is designed to make rotation zero-downtime:

1. **Issue a new key** from the Fundamental.tech dashboard
2. **Update the environment variable** in your secret manager (AWS Secrets Manager, HashiCorp Vault, etc.)
3. **Restart your service** (or trigger a config reload) -- the new `Fundamental()` call picks up the updated variable
4. **Revoke the old key** from the dashboard after confirming the new one is in use

Because the SDK reads `FUNDAMENTAL_API_KEY` at client construction time (not at import time), a simple process restart is sufficient. You do not need to redeploy code.

The pattern below shows how to build a client that validates its key on startup, so rotation failures are caught immediately rather than at the first API call hours later.

In [4]:
def build_validated_client(env="dev"):
    """
    Build a client and immediately validate the key by listing models.
    Raises on startup if the key is wrong, rather than failing silently
    during the first real API call.
    """
    c = build_client(env=env)

    try:
        _ = c.models.list()  # Fast, read-only call to validate the key
        print(f"[{env}] Key validated. Client ready.")
    except Exception as e:
        raise ValueError(
            f"[{env}] API key validation failed. "
            f"Check FUNDAMENTAL_API_KEY_{env.upper()} or rotate the key. "
            f"Error: {e}"
        ) from e

    return c


# Validate the production client on startup
try:
    validated = build_validated_client(env="prod")
    set_client(validated)
except ValueError as e:
    print(f"Startup auth failure: {e}")

[prod] Key validated. Client ready.


---

## Part 2: Model Lifecycle with `client.models`

### What Gives You

In Module 1 you learned that the `trained_model_id_` is your permanent handle to a trained model. the SDK expands on that with a full lifecycle API on `client.models`:

| Method | What it does |
|--------|--------------|
| `client.models.list()` | List every trained model in your account |
| `client.models.get(model_id)` | Look up a specific model and read its attributes |
| `client.models.set_attributes(model_id, attributes)` | Tag a model with key-value metadata (version, team, stage, owner...) |
| `client.models.delete(model_id)` | Permanently delete a model |

`set_attributes` is also available directly on the estimator: `clf.set_attributes({...})` after a successful `fit` is equivalent to calling `client.models.set_attributes(clf.trained_model_id_, {...})`.

The lifecycle this enables is straightforward: **tag on fit, look up on deploy, delete on retire.** A model with no `stage` attribute is an orphan; one tagged `stage="experiment"` and older than 30 days is a deletion candidate; one tagged `stage="production"` is locked. The discipline scales — you can have hundreds of models in your account and still find the one you need in seconds.

A convention worth adopting: set `version`, `team`, `stage`, and `owner` the moment `fit` returns, every time — then add domain tags as needed. Untagged models become orphans.

In [5]:
# Tagging the model from Module 6 with production-style attributes.
# Two equivalent paths: via the estimator, or via client.models directly.

# Path 1: via the loaded estimator
nexus.set_attributes({
    "version": "1.0",
    "team": "ml-ops",
    "stage": "production",
    "owner": "credit-risk-pod",
    "feature_set": "top_15_enriched",
})
print(f"Attributes set on model {nexus.trained_model_id_}")

# Path 2: via client.models (works without an estimator instance)
# client.models.set_attributes(CURRENT_MODEL_ID, {
#     "version": "1.0",
#     "team": "ml-ops",
#     "stage": "production",
# })

# Read them back via client.models.get()
metadata = client.models.get(CURRENT_MODEL_ID)
print(f"\nModel metadata for {metadata.trained_model_id}:")
for k, v in metadata.attributes.items():
    print(f"  {k:<15} = {v}")

Attributes set on model b135ba5b-da0c-457c-8d23-a2f58505b79f



Model metadata for b135ba5b-da0c-457c-8d23-a2f58505b79f:
  name            = 
  description     = 
  version         = 1.0
  team            = ml-ops
  stage           = production
  owner           = credit-risk-pod
  feature_set     = top_15_enriched


In [6]:
# Pruning stale experimental models with a safe-by-default pattern:
# 1) list, 2) filter by tag, 3) confirm count, 4) delete.

THIRTY_DAYS_AGO = pd.Timestamp.now() - pd.Timedelta(days=30)

candidates = []
for m in client.models.list():
    md = client.models.get(m.trained_model_id if hasattr(m, "trained_model_id") else m)
    attrs = md.attributes or {}
    # Only consider models tagged as experiments (never delete unlabeled or production models)
    if attrs.get("stage") == "experiment":
        candidates.append(md.trained_model_id)

print(f"Found {len(candidates)} experimental models eligible for cleanup.")

# In a real script you would assert on the count, log the IDs, and then
# delete in a loop. We DO NOT actually call delete() in this workshop run --
# uncomment when you want to actually prune.
#
# for model_id in candidates:
#     resp = client.models.delete(model_id)
#     print(f"Deleted {resp.trained_model_id}: {resp.message}")
print("\n(Skipping actual deletion in workshop run; uncomment the loop above to execute.)")

Found 0 experimental models eligible for cleanup.

(Skipping actual deletion in workshop run; uncomment the loop above to execute.)


### Choosing Your Mode

Across the workshop you've seen the two training modes supports:

| Mode | Best for | Trade-off |
|------|----------|-----------|
| `speed` | Fast iteration, prototyping, large datasets where training time matters | Sacrifices some accuracy for speed |
| `quality` | Production models, final submissions, when every basis point matters | Longer training time, evaluates more features for the best fit |

A common workflow: prototype with `speed`, validate with `quality`, ship with `quality`. Tag the production-bound model with `stage="production"` (Part 2) so future-you can identify it without reading commit history.

> **Pinning a model version.** Both estimators also accept a `model_version` constructor parameter to pin training to a specific SDK model version — e.g. `NEXUSClassifier(mode="quality", model_version="0.1.7")`. Leave it unset to use the latest; call `client.models.get_supported_versions()` to list what is available. Handy when you need reproducible retraining across a fleet of models.

---

## Part 3: Model Inventory

### Keeping Track of Your Models

Every `fit()` or `submit_fit_task()` call creates a new model in your account. After a few months of active development, you can easily accumulate dozens of models — experimental runs, feature-set variants, A/B variants — with no obvious way to tell which one is which.

`client.models.list()` returns the full list. Combined with `client.models.get(model_id)`, you can read each model's attributes (the metadata you set via `set_attributes` in Part 2) and build a complete picture of what lives in your account.

This is the cornerstone of model lifecycle hygiene. Train it, tag it, list it, look it up, retire it — every step has a single API call.

In [7]:
# Build a model inventory from client.models.list() + client.models.get()
models = client.models.list()

rows = []
for m in models[:25]:  # cap at 25 for the workshop run; remove the slice in production
    # m can be a dict, a string, or an object — handle both
    model_id = m.trained_model_id if hasattr(m, "trained_model_id") else (m if isinstance(m, str) else m.get("trained_model_id"))
    try:
        md = client.models.get(model_id)
        attrs = md.attributes or {}
    except Exception:
        attrs = {}
    rows.append({
        "model_id": model_id,
        "version": attrs.get("version"),
        "team": attrs.get("team"),
        "stage": attrs.get("stage"),
        "owner": attrs.get("owner"),
    })

inventory = pd.DataFrame(rows)
print(f"Total models in account (showing up to 25): {len(inventory)}")
if not inventory.empty:
    print(inventory.to_string(index=False))
else:
    print("No models found.")

Total models in account (showing up to 25): 25
                            model_id version     team      stage           owner
8a10b687-bc0b-498b-bf4a-eebb06209607    None     None       None            None
b135ba5b-da0c-457c-8d23-a2f58505b79f     1.0   ml-ops production credit-risk-pod
966ff1c1-0475-4140-96db-516e4640c5d6    None     None       None            None
67ce8f2a-87a5-42a6-9e11-3f2c21eb7f1a    None     None       None            None
86f166b6-8c18-4cb3-8144-7950bd21a4eb    None     None       None            None
c4b1d314-bffd-43bd-9485-5c91d4256d75    None     None       None            None
d7caae7c-941d-4746-ae42-fdaa0508a8de    None     None       None            None
80b98f16-cd48-46e8-a7a6-bba2cfab5df8    None     None       None            None
61fd865b-32a0-4b05-b1aa-2c5315f26b8f    None     None       None            None
e3abeb3e-8b8e-45b3-9f76-a086028d6ab3    None     None       None            None
4ae72528-490d-473e-984c-032e0d4d0b72    None     None       No

### Best Practices for Model Lifecycle Management

The four-method API gives you everything you need; the discipline is in how you use it.

**Tag on fit.** Every `fit()` call should be followed immediately by `clf.set_attributes({...})`. At minimum: `version`, `stage` (`experiment`, `staging`, `production`), `team`, `owner`. Untagged models are orphans waiting to happen.

**Look up on deploy.** Production deployment code should call `client.models.get(model_id)` and verify the `stage` attribute is `"production"` before loading the model. This catches the "wrong-environment model ID" class of bug at the boundary, not at first prediction.

**Maintain a local registry too.** `client.models.list()` is the source of truth, but a local `model_registry.json` (one record per model with `model_id`, `trained_at`, `dataset`, `notes`) gives you faster lookups and survives API outages.

```python
import json, datetime

record = {
    "model_id": nexus.trained_model_id_,
    "task": "classification",
    "mode": "quality",
    "trained_at": datetime.datetime.now(datetime.UTC).isoformat(),
    "dataset": "credit_risk",
    "notes": "production candidate after feature refresh",
}

with open("model_registry.json", "a") as f:
    f.write(json.dumps(record) + "\n")
```

**Prune monthly.** Run `client.models.list()` once a month, filter by `stage="experiment"` and age > 30 days, and `client.models.delete(model_id)` the survivors. Production models are locked by their `stage` tag; experiments cost nothing to keep but clutter your view.

In [8]:
# The complete production startup sequence

def initialize_production(env="prod", model_id=None):
    """
    Complete production environment initialization.
    Run this at the start of any production pipeline.

    1. Build and validate the client
    2. Look up the model and verify its stage attribute
    3. Load the model into a NEXUSClassifier instance
    """
    # 1. Build and validate the client
    client = build_validated_client(env=env)
    set_client(client)

    if not model_id:
        raise RuntimeError(
            "No model_id provided. Pass the production model ID explicitly -- "
            "it should come from your model registry or deployment config."
        )

    # 2. Verify the model is tagged for production before loading it
    md = client.models.get(model_id)
    stage = (md.attributes or {}).get("stage")
    if stage != "production":
        raise RuntimeError(
            f"Refusing to load model {model_id}: stage='{stage}' (expected 'production'). "
            f"Tag this model via client.models.set_attributes() or use a different model_id."
        )

    # 3. Load it
    nexus = NEXUSClassifier()
    nexus.load_model(model_id)
    print(f"[{env}] Production model loaded: {model_id}")
    print(f"        attributes: {md.attributes}")

    return client, nexus


print("initialize_production defined.")
print()
print("Typical production startup:")
print('  client, nexus = initialize_production(env="prod", model_id=PROD_MODEL_ID)')
print("  proba = nexus.predict_proba(X_new)")

initialize_production defined.

Typical production startup:
  client, nexus = initialize_production(env="prod", model_id=PROD_MODEL_ID)
  proba = nexus.predict_proba(X_new)


### Capstone: the full production loop

Everything from Modules 6–8 in one pass. In the scaffold below, write the code for each step:

1. Build a validated client (`build_validated_client`) for an environment of your choice.
2. Load the Module 6 model into a fresh `NEXUSClassifier` (or train a new one).
3. Tag it: `version`, `team`, `stage`, `owner`.
4. Write a registry entry — model id plus your tags — to `module_08_capstone_registry.json`.
5. Confirm the model appears in `client.models.list()`.

In [9]:
# Step 1: client = build_validated_client(...)
# Step 2: model = NEXUSClassifier().load_model(...)
# Step 3: model.set_attributes({...})
# Step 4: write the registry entry to module_08_capstone_registry.json
# Step 5: confirm the model id appears in client.models.list()

**Optional — go deeper.** One reference solution:

In [10]:
cap_client = build_validated_client(env="prod")

cap_model = NEXUSClassifier()
cap_model.load_model(CURRENT_MODEL_ID)
cap_model.set_attributes({
    "version": "1.0", "team": "workshop", "stage": "production", "owner": "you",
})

entry = {"model_id": CURRENT_MODEL_ID, "task": "classification", "stage": "production"}
with open("module_08_capstone_registry.json", "w") as f:
    json.dump([entry], f, indent=2)

inventory_ids = [
    m.trained_model_id if hasattr(m, "trained_model_id") else (m if isinstance(m, str) else m.get("trained_model_id"))
    for m in cap_client.models.list()
]
print(f"Tagged and registered. In inventory: {CURRENT_MODEL_ID in inventory_ids}")

[prod] Key validated. Client ready.


Tagged and registered. In inventory: True


---

## Summary

| Topic | Key API / Pattern |
|-------|------------------|
| Multi-environment keys | `FUNDAMENTAL_API_KEY_{ENV}` env vars + `build_client(env)` |
| Deployment target | `api_url` per environment (or the `FUNDAMENTAL_API_URL` env var) |
| Key validation at startup | `client.models.list()` as a fast read-only health check |
| Tag a model | `clf.set_attributes({...})` or `client.models.set_attributes(model_id, {...})` |
| Look up a model | `client.models.get(model_id)` — returns `TrainedModelMetadata` |
| Inventory | `client.models.list()` — full list of trained models in your account |
| Prune | `client.models.delete(model_id)` — destructive; gate on tag + age |
| Model persistence | `clf.trained_model_id_` — save it, because it is your only handle |

---

## Key Takeaways

**Multi-environment key management lets you safely separate dev, staging, and production.** `FUNDAMENTAL_API_KEY_{ENV}` per environment, read at client construction time. Rotation is a variable update, not a code change.

**Each environment gets its own `api_url`.** Pass the matching endpoint to `Fundamental(api_key=..., api_url=...)`, or set `FUNDAMENTAL_API_URL` in the environment. Dev and prod never share a target. No URL is hardcoded in source.

**The `client.models` API is the model lifecycle layer.** `list`, `get`, `set_attributes`, `delete`. Train it, tag it, list it, look it up, retire it — every step has a single call.

**Tag on fit, look up on deploy, prune monthly.** `clf.set_attributes()` immediately after training, `client.models.get()` before promoting to production, `client.models.delete()` for stale experiments older than 30 days. Untagged models are orphans.

**Model IDs are everything.** `clf.trained_model_id_` is the only handle you have on a trained model. Log it immediately after training, store it in a registry, and treat it like a production artifact.

---

**Congratulations — you have completed the NEXUS Onboarding Workshop.**

You started with a single `fit()` call on numeric features. You are leaving with async pipelines, resilient retry logic, full diagnostic instrumentation, and a production model-lifecycle playbook. The patterns in this workshop are production-tested. Use them as starting points, adapt them to your team's conventions, and build on top of them.

Welcome to Fundamental.tech.